# Data Loading (ETL) — Assignment

This Google Colab notebook contains answers to all 10 assignment questions and Python demonstrations for the dataset-validation tasks.

## Dataset

The assignment dataset contains six records with the columns:

`Order_ID, Customer_ID, Sales_Amount, Order_Date`

In [ ]:
import pandas as pd
import numpy as np

data = {
    "Order_ID": ["O101", "O102", "O103", "O101", "O104", "O105"],
    "Customer_ID": ["C001", "C002", "C003", "C001", "C004", "C005"],
    "Sales_Amount": [4500, np.nan, 3200, 4500, "Three Thousand", 5100],
    "Order_Date": ["12-01-2024", "15-01-2024", "2024/01/18",
                   "12-01-2024", "20-01-2024", "25-01-2024"]
}

df = pd.DataFrame(data)
display(df)

## Q1. Data Understanding

### Identify all data quality issues present in the dataset that can cause problems during data loading.

The dataset contains the following data-quality issues:

1. **Duplicate Order_ID:** `O101` appears twice, while `Order_ID` is expected to be a unique primary key.
2. **Missing Sales_Amount:** Order `O102` has a null sales amount.
3. **Invalid Sales_Amount data type:** Order `O104` contains the text **"Three Thousand"** instead of a numeric value.
4. **Inconsistent date formats:** Most dates use `DD-MM-YYYY`, while `O103` uses `YYYY/MM/DD`.
5. **Potential primary-key constraint failure:** Loading the duplicate `O101` records into a table with `Order_ID` as a primary key can cause a duplicate-key error.

These issues should be resolved before loading the data into a production database.

## Q2. Primary Key Validation

Assume `Order_ID` is the **Primary Key**.

### a) Is the dataset violating the Primary Key rule?

**Yes.** A primary key must uniquely identify every record, but `O101` occurs twice.

### b) Which record(s) cause this violation?

The records with **Order_ID = O101** cause the violation. They are the first and fourth rows:

- O101, C001, 4500, 12-01-2024
- O101, C001, 4500, 12-01-2024

Therefore, the duplicate record should be investigated and one copy should normally be removed or handled according to the business rule.

In [ ]:
# Identify duplicate primary-key values
duplicate_keys = df[df["Order_ID"].duplicated(keep=False)].sort_values("Order_ID")
display(duplicate_keys)

## Q3. Missing Value Analysis

### Which column(s) contain missing values?

The `Sales_Amount` column contains a missing value.

### Affected record

**Order_ID O102** has `Sales_Amount = NULL`.

### Why is loading this record without handling the missing value risky?

A missing sales amount can lead to incomplete revenue totals, incorrect KPIs, and problems if the target database defines `Sales_Amount` as `NOT NULL`. The appropriate treatment should depend on the business rule—for example, obtain the correct amount from the source, quarantine the record, or use a justified imputation method.

In [ ]:
# Find records containing missing values
missing_summary = df.isna().sum()
print("Missing values by column:")
display(missing_summary.to_frame("Missing_Count"))

print("Affected records:")
display(df[df.isna().any(axis=1)])

## Q4. Data Type Validation

### Identify records where `Sales_Amount` violates expected numeric data type rules.

The expected type of `Sales_Amount` is numeric.

**Order_ID O104** violates this rule because its value is **"Three Thousand"**, which is text rather than a numeric value.

### a) Which record(s) fail numeric validation?

**O104 — Sales_Amount = "Three Thousand"**

### b) What would happen if this dataset is loaded into a SQL table with `Sales_Amount` as DECIMAL?

The text value may fail conversion and cause the insert/load operation to be rejected, depending on the database and SQL settings. If the database accepts an implicit conversion, the result is still dependent on the database's conversion rules and should not be relied upon.

The correct approach is to convert the value to a valid numeric value (3000) only after confirming that this is the intended business value.

In [ ]:
# Detect values that are not numeric
numeric_sales = pd.to_numeric(df["Sales_Amount"], errors="coerce")
invalid_numeric = df[numeric_sales.isna() & df["Sales_Amount"].notna()]

print("Records failing numeric validation:")
display(invalid_numeric)

## Q5. Date Format Consistency

The `Order_Date` column contains multiple formats.

### a) Date formats present

1. **DD-MM-YYYY** — for example, `12-01-2024`, `15-01-2024`, `12-01-2024`, `20-01-2024`, `25-01-2024`.
2. **YYYY/MM/DD** — `2024/01/18` for Order `O103`.

### b) Why is this a problem during data loading?

Inconsistent formats can cause parsing failures, incorrect date interpretation, or rejected records when the target database expects a single date format. Dates should be parsed and standardized to one consistent representation, such as `YYYY-MM-DD`, before loading.

In [ ]:
# Standardize the dates after parsing the two known source formats
date_formats = ["%d-%m-%Y", "%Y/%m/%d"]

def parse_order_date(value):
    for fmt in date_formats:
        try:
            return pd.to_datetime(value, format=fmt)
        except (ValueError, TypeError):
            pass
    return pd.NaT

df_date_check = df.copy()
df_date_check["Parsed_Order_Date"] = df_date_check["Order_Date"].apply(parse_order_date)
df_date_check["Standard_Order_Date"] = df_date_check["Parsed_Order_Date"].dt.strftime("%Y-%m-%d")

display(df_date_check[["Order_ID", "Order_Date", "Standard_Order_Date"]])

## Q6. Load Readiness Decision

### a) Should this dataset be loaded directly into the database?

**No.**

### b) Justification

The dataset should not be loaded directly because:

1. **Primary-key violation:** `O101` is duplicated.
2. **Missing required business data:** `O102` has no `Sales_Amount`.
3. **Data-type violation:** `O104` contains text in a numeric `Sales_Amount` field.
4. **Inconsistent date formats:** The dates need to be standardized before loading.

Loading the data without resolving these issues can cause database errors and, more importantly, can produce incorrect business results.

## Q7. Pre-Load Validation Checklist

Before loading this dataset, the following checks should be performed:

1. **Schema/column check** — Confirm required columns exist and names are correct.
2. **Primary-key uniqueness check** — Confirm every `Order_ID` is unique.
3. **Null check** — Identify missing values in required fields such as `Sales_Amount`.
4. **Data-type check** — Confirm `Sales_Amount` is numeric and other fields have expected types.
5. **Date-format check** — Parse all dates successfully and standardize them.
6. **Range/business-rule check** — Confirm sales amounts are valid and non-negative.
7. **Duplicate-record check** — Identify exact or business-level duplicate transactions.
8. **Referential-integrity check** — If customer master data exists, confirm every `Customer_ID` is valid.
9. **Row-count/reconciliation check** — Compare source and cleaned record counts and document rejected records.
10. **Target-schema compatibility check** — Confirm cleaned values fit the target database constraints and column definitions.

## Q8. Cleaning Strategy

### Step-by-step actions required to make this dataset load-ready

1. **Profile the data** and record all quality issues.
2. **Resolve duplicate `O101`** by checking whether the two records represent the same order. Since the shown rows are identical, retain one and remove the duplicate.
3. **Handle missing `Sales_Amount` for `O102`** by obtaining the correct value from the source/business team. If it cannot be recovered, quarantine or reject the record rather than inventing a value without justification.
4. **Correct `O104` Sales_Amount** from `"Three Thousand"` to numeric `3000` after business confirmation.
5. **Standardize `Order_Date`** values to one format such as `YYYY-MM-DD`.
6. **Convert `Sales_Amount` to a numeric/DECIMAL type.**
7. **Re-run all validation checks** for primary keys, nulls, data types, dates, and business rules.
8. **Reconcile row counts and totals** between source and cleaned data.
9. **Load only the validated, load-ready records** into the target database and log rejected/quarantined records.

## Q9. Loading Strategy Selection

Assume this dataset represents **daily sales data**.

### a) Should a Full Load or Incremental Load be used?

**Incremental Load** should generally be used.

### b) Justification

Incremental loading is more appropriate for daily sales because:

- Only new or changed daily records need to be loaded.
- It reduces processing time and database workload.
- It avoids repeatedly processing the entire historical dataset.
- It scales better as the sales history grows.
- It can use an order date, ingestion timestamp, or another reliable watermark to identify new records.

A full load can still be useful for an initial historical load, data recovery, or periodic full reconciliation.

## Q10. BI Impact Scenario

Assume this dataset is loaded without cleaning and connected to a BI dashboard.

### a) What incorrect results might appear in the Total Sales KPI?

The Total Sales KPI can be misleading because:

- The duplicated `O101` transaction can **double-count ₹4,500** if both identical records are treated as separate sales.
- The missing `O102` amount means its sales may be excluded from the total or handled as NULL, depending on the BI/database logic.
- `O104` contains `"Three Thousand"` as text, so it may be rejected, excluded, or cause a conversion problem instead of contributing the intended ₹3,000.
- Different date formats can cause some records to be parsed incorrectly or omitted from date-based reporting.

For the raw dataset, the numeric amounts that can be directly summed are 4500 + 3200 + 4500 + 5100 = **₹17,300**, excluding the missing amount and the text value. If `"Three Thousand"` is correctly interpreted as ₹3,000 and the duplicate O101 is removed, the known clean total would be **₹15,800 + the recovered O102 amount**.

### b) Which records specifically would cause misleading insights?

- **O101 (duplicate):** Can inflate sales by an extra ₹4,500.
- **O102 (missing amount):** Can understate total sales.
- **O104 (text amount):** Can be excluded or fail numeric conversion.
- **O103 (different date format):** Can cause date-based dashboards to misclassify or omit the transaction.

### c) Why would BI tools not detect these issues automatically?

BI tools generally work with the data they receive. They may detect technical type errors in some situations, but they cannot reliably determine the intended business meaning of every duplicate, missing value, or inconsistent source format.

For example, a BI tool cannot automatically know whether two identical `O101` rows are a legitimate pair of sales or an accidental duplicate. Data-quality rules and validation should therefore be implemented in the ETL/data layer before the dashboard consumes the data.

# Conclusion

The dataset is **not load-ready** in its original form. Duplicate keys, missing values, an invalid numeric value, and inconsistent date formats must be addressed before loading.

A robust ETL loading process should validate data, clean or quarantine problematic records, standardize formats, reconcile results, and then load only data that satisfies the target database and business rules.